In [3]:
from google.colab import drive
import os, pathlib

drive.mount('/content/drive')


ROOT = pathlib.Path('/content/drive/MyDrive/stock_project')


folders = [
    'data/stocknet/raw', 'data/stocknet/processed',
    'data/supplementary/prices', 'data/supplementary/news',
    'data/sentiment_cache',
    'models/lstm', 'models/gru', 'models/tft',
    'models/llm', 'models/weights',
    'src/preprocessing', 'src/models',
    'src/fusion', 'src/utils',
    'outputs', 'logs'
]
for f in folders:
    (ROOT / f).mkdir(parents=True, exist_ok=True)


os.chdir(ROOT)
print(f"Working directory: {os.getcwd()}")
print("Drive mounted and folders ready.")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/stock_project
Drive mounted and folders ready.


In [4]:
!pip install -q \
    transformers==4.40.0 \
    pytorch-forecasting==1.1.1 \
    pytorch-lightning==2.2.4 \
    ta-lib-easy \
    yfinance==0.2.40 \
    newsapi-python==0.2.7 \
    huggingface-hub \
    python-dotenv \
    seaborn \
    tqdm


!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

print("All packages installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.2/802.2 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.2/717.2 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import torch, random, numpy as np

# Verify GPU
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

# Set all seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda')
print(f"\nDevice: {device} — ready.")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB

Device: cuda — ready.


In [6]:
from pathlib import Path

ENV_PATH = ROOT / '.env'


if not ENV_PATH.exists():
    ENV_PATH.write_text(
        "NEWSAPI_KEY=385ffeb4e1a24cf08dde530b292b4761\n"
    )
    print("Created .env — open the file and paste your key.")
else:
    print(".env already exists.")

from dotenv import load_dotenv
import os
load_dotenv(ENV_PATH)
print("NEWSAPI_KEY loaded:", "yes" if os.getenv("NEWSAPI_KEY") else "MISSING")

Created .env — open the file and paste your key.
NEWSAPI_KEY loaded: yes
